<center><span style="color:#336699">
CAP-394-3 - Introdução à Ciência de Dados</span></center>
<hr style="border:2px solid #0077b9;">

<br/>

<div style="text-align: center;font-size: 200%;"> 
Trabalho Final 
 <br/>
</div>


<br/>

<div style="text-align: center;font-size: 90%;">
    Carla Aparecida de Almeida Paula Almeida,Sofia Sena Tavares  <sup><a href=""></i></a></sup>
    <br/><br/>
     Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Atualização: 2 de Agosto de 2026
</div>

<br/>

<div style="text-align: justify;  margin-left: 25%; margin-right: 25%;">
    <b>Resumo.</b> Essa segunda  etapa refere-se a extração da imagem via stac.
</div>

<br/>


In [7]:
from pystac_client import Client

# Servidor STAC aberto e sem necessidade de token
STAC_URL = "https://earth-search.aws.element84.com/v1"
catalog = Client.open(STAC_URL)

collection_id = "sentinel-2-l2a"  # Nome da coleção no Earth Search
bands = ["red", "nir", "nir08"]  # Nomes das bandas equivalentes no Sentinel AWS

In [9]:
import pandas as pd
import numpy as np
import xarray as xr
from pystac_client import Client
import stackstac

# =====================================================================
# 1. CARREGAR PONTOS DO CSV
# =====================================================================
csv_path = "../data/dataset_treino_ml.csv"
df_pontos = pd.read_csv(csv_path)

if "CLASS" in df_pontos.columns and "label" not in df_pontos.columns:
    df_pontos["label"] = df_pontos["CLASS"]

print(f"Total de pontos carregados: {len(df_pontos)}")

# =====================================================================
# 2. CONECTAR AO STAC EARTH SEARCH (AWS - Gratuito e Aberto)
# =====================================================================
STAC_URL = "https://earth-search.aws.element84.com/v1"
catalog = Client.open(STAC_URL)

min_lon, max_lon = df_pontos["longitude"].min() - 0.05, df_pontos["longitude"].max() + 0.05
min_lat, max_lat = df_pontos["latitude"].min() - 0.05, df_pontos["latitude"].max() + 0.05
bbox = [min_lon, min_lat, max_lon, max_lat]

collection_id = "sentinel-2-c1-l2a"  # Coleção Sentinel-2 L2A atualizada na AWS
time_range = "2023-01-01/2023-12-31"

print(f"\nConsultando o catálogo STAC AWS ({collection_id})...")
search = catalog.search(
    collections=[collection_id],
    bbox=bbox,
    datetime=time_range,
    query={"eo:cloud_cover": {"lt": 30}}  # Filtrar cenas com menos de 30% de nuvens na cena total
)

items = list(search.items())
print(f"Total de itens STAC encontrados: {len(items)}")

if len(items) == 0:
    raise ValueError("Nenhum item encontrado. Tente aumentar a margem do bbox ou o intervalo de datas.")

# =====================================================================
# 3. MONTAR O CUBO DE DADOS VIA STACKSTAC
# =====================================================================
# red = B04, nir = B08, scl = Máscara de classificação de cena
bands = ["red", "nir", "scl"]

cube = stackstac.stack(
    items,
    assets=bands,
    bounds=bbox,
    epsg=4326,
    resolution=0.0001  # ~10m
)

# =====================================================================
# 4. TRATAR MÁSCARA DE NUVENS (SCL) E CALCULAR NDVI
# =====================================================================
print("\nAplicando máscara de nuvens e calculando NDVI...")

# No Sentinel-2 SCL: 4 = Vegetação, 5 = Solo Exposto, 6 = Água
scl = cube.sel(band="scl")
clear_mask = (scl == 4) | (scl == 5) | (scl == 6)

# Aplicar a máscara nas bandas espectrais
red = cube.sel(band="red").where(clear_mask)
nir = cube.sel(band="nir").where(clear_mask)

# Calcular o NDVI diretamente no cubo
ndvi = (nir - red) / (nir + red)

# Combinar as bandas em um novo DataArray organizado
cube_processed = xr.concat([red, nir, ndvi], dim="band")
cube_processed["band"] = ["red", "nir", "ndvi"]

# =====================================================================
# 5. EXTRAIR PONTOS DE FORMA VETORIZADA EM LOTE
# =====================================================================
print("\nExtraindo dados dos pontos em lote...")

x_coords = xr.DataArray(df_pontos["longitude"].values, dims="ponto")
y_coords = xr.DataArray(df_pontos["latitude"].values, dims="ponto")

# Selecionar todos os pontos no cubo
amostras = cube_processed.sel(x=x_coords, y=y_coords, method="nearest")

print("Processando requisições em memória RAM...")
amostras_computed = amostras.compute()

# Interpolar buracos deixados por nuvens ao longo do tempo
amostras_interp = amostras_computed.interpolate_na(dim="time", method="linear")

# =====================================================================
# 6. ENGENHARIA DE ATRIBUTOS (MÉTRICAS TEMPORAIS)
# =====================================================================
print("\nGerando matriz de atributos para Machine Learning...")

registros_extraidos = []
target_bands = ["red", "nir", "ndvi"]

for idx in range(len(df_pontos)):
    ponto_row = df_pontos.iloc[idx]
    ponto_ds = amostras_interp.isel(ponto=idx)
    
    dict_features = {
        "longitude": ponto_row["longitude"],
        "latitude": ponto_row["latitude"],
        "label": ponto_row["label"]
    }
    
    for band_name in target_bands:
        serie = ponto_ds.sel(band=band_name).values
        
        # Limpar valores restantes
        if np.isnan(serie).all():
            serie = np.zeros_like(serie)
        else:
            mediana_val = np.nanmedian(serie)
            serie = np.nan_to_num(serie, nan=mediana_val)
        
        # Atributos estatísticos da série temporal anual
        dict_features[f"{band_name}_mean"] = float(np.mean(serie))
        dict_features[f"{band_name}_std"] = float(np.std(serie))
        dict_features[f"{band_name}_min"] = float(np.min(serie))
        dict_features[f"{band_name}_max"] = float(np.max(serie))
        dict_features[f"{band_name}_p10"] = float(np.percentile(serie, 10))
        dict_features[f"{band_name}_p90"] = float(np.percentile(serie, 90))
        dict_features[f"{band_name}_amp"] = float(np.max(serie) - np.min(serie))

    registros_extraidos.append(dict_features)

# =====================================================================
# 7. SALVAR CSV FINAL
# =====================================================================
df_features_final = pd.DataFrame(registros_extraidos)
caminho_csv = "dataset_stac_aws_deter_floresta.csv"
df_features_final.to_csv(caminho_csv, index=False)

print(f"\nSucesso! Arquivo salvo em: {caminho_csv}")
print(f"Shape da tabela de treino: {df_features_final.shape}")
print(df_features_final.head())

Total de pontos carregados: 200

Consultando o catálogo STAC AWS (sentinel-2-c1-l2a)...


/opt/anaconda3/envs/geo_env/lib/python3.11/site-packages/pystac/extensions/storage.py:724: UserWarning: Could not parse bucket/account from href. The following assets were not migrated: ['red', 'green', 'blue', 'visual', 'nir', 'swir22', 'rededge2', 'rededge3', 'rededge1', 'swir16', 'wvp', 'nir08', 'scl', 'aot', 'coastal', 'nir09', 'cloud', 'snow', 'preview', 'granule_metadata', 'tileinfo_metadata', 'product_metadata', 'thumbnail']
  warnings.warn(
/opt/anaconda3/envs/geo_env/lib/python3.11/site-packages/pystac/extensions/storage.py:724: UserWarning: Could not parse bucket/account from href. The following assets were not migrated: ['nir08', 'preview', 'thumbnail', 'green', 'product_metadata', 'aot', 'tileinfo_metadata', 'swir16', 'granule_metadata', 'red', 'wvp', 'cloud', 'blue', 'nir', 'swir22', 'snow', 'visual', 'rededge2', 'rededge3', 'rededge1', 'scl', 'coastal', 'nir09']
  warnings.warn(


Total de itens STAC encontrados: 45

Aplicando máscara de nuvens e calculando NDVI...

Extraindo dados dos pontos em lote...
Processando requisições em memória RAM...

Gerando matriz de atributos para Machine Learning...

Sucesso! Arquivo salvo em: dataset_stac_aws_deter_floresta.csv
Shape da tabela de treino: (200, 24)
   longitude  latitude      label  red_mean   red_std  red_min  red_max  \
0 -59.909459 -6.052428  Disturbio  0.065415  0.011518   0.0515   0.1004   
1 -59.955452 -6.097412  Disturbio  0.035298  0.020043   0.0183   0.0903   
2 -59.936309 -6.079300  Disturbio  0.058634  0.022574   0.0286   0.0958   
3 -59.905957 -6.044969  Disturbio  0.067899  0.010296   0.0504   0.0866   
4 -59.911097 -6.123147  Disturbio  0.038499  0.026542   0.0204   0.1448   

   red_p10   red_p90  red_amp  ...   nir_p10   nir_p90  nir_amp  ndvi_mean  \
0  0.05400  0.081050   0.0489  ...  0.099100  0.291000   0.2093   0.471265   
1  0.01920  0.070884   0.0720  ...  0.265200  0.346175   0.1844   0.796